# GPU-Only Training: deepseek-coder-v2:16b for Cline Tools (FIXED)

**Run this notebook in Kaggle or Google Colab** (Free tier T4 compatible)

## What this does:
1. Loads pre-prepared training data from HF Hub (no CPU-intensive data prep)
2. Loads `deepseek-coder-v2-lite-base` (16B) with **4-bit quantization** directly to GPU
3. Trains on Cline's 9 native tool calling format using GPU
4. **Pushes checkpoints to HF Hub during training** for resume after runtime stops
5. Auto-resumes from latest HF Hub checkpoint on restart

## Prerequisites:
- Kaggle or Google Colab (Free tier T4 works!)
- Hugging Face token with **Write** permission on repositories
- Training data already uploaded to HF Dataset repo via WSL preparation
- ~12GB VRAM needed (T4 16GB with 4-bit quantization)

## Workflow:
1. **WSL**: Run `python scripts/prepare_and_upload_data.py` (CPU-intensive data prep)
2. **GPU **: Run this notebook (GPU-intensive training)
3. **WSL**: After training, run merge_and_export.py

---


In [ ]:
# @title 1. Configuration - FILL THESE IN
# ============================================
# CRITICAL: Must be set BEFORE any torch/CUDA import.
# Without this, torch sees 2 GPUs on Kaggle and Trainer wraps the model with
# nn.DataParallel, which is incompatible with bitsandbytes 4-bit quantization
# → causes CUBLAS_STATUS_NOT_INITIALIZED at the first training step.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Force single GPU — MUST be first

HF_USERNAME = "COleo"      # Your HF username
HF_TOKEN = "xxxxxxxxx"   # Your HF token (Write permission)
DATASET_REPO = f"{HF_USERNAME}/cline-tools-data"
MODEL_REPO = f"{HF_USERNAME}/deepseek-coder-v2-16b-tools"

# Training config (single T4, 16GB VRAM, 4-bit quantization)
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 5
GRAD_ACCUM = 32            # Effective batch = 32
EPOCHS = 2
LEARNING_RATE = 2e-4

# Checkpointing to HF Hub
PUSH_EVERY_N_STEPS = 5   # Push full checkpoint to HF Hub every N steps
RESUME_FROM_HUB = True     # Auto-resume from latest HF Hub checkpoint on restart

print(f"HF Username: {HF_USERNAME}")
print(f"Dataset repo: {DATASET_REPO}")
print(f"Model repo: {MODEL_REPO}")
print(f"Max seq length: {MAX_SEQ_LENGTH}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Push checkpoints every: {PUSH_EVERY_N_STEPS} steps")
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']} (single GPU enforced)")


HF Username: COleo
Dataset repo: COleo/cline-tools-data
Model repo: COleo/deepseek-coder-v2-16b-tools
Max seq length: 2048
Effective batch: 160
Push checkpoints every: 5 steps
CUDA_VISIBLE_DEVICES=0 (single GPU enforced)


In [2]:
# @title 2. Verify GPU & Clear CUDA Context (CRITICAL FIX)
!nvidia-smi

import torch
import gc
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. This notebook requires GPU for training.")
print("✅ GPU available")

# CRITICAL: Clear any existing CUDA context to prevent CUBLAS_STATUS_NOT_INITIALIZED
# This must be done BEFORE any model loading
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()
# Force reinitialize CUDA context
torch.cuda.init()
print("✅ CUDA context cleared and reinitialized")

# Set memory fraction to prevent OOM and fragmentation (must be set before model loading)
torch.cuda.set_per_process_memory_fraction(0.95)
print("✅ Set GPU memory fraction to 95%")

# Clean up any conflicting packages (Colab/Kaggle specific)
!pip uninstall -y cudf-cu12 cuml-cu12 dask-cuda ucxx-cu12 distributed-ucxx-cu12 2>/dev/null || true
!pip uninstall -y numba numba-cuda cuda-core 2>/dev/null || true
!pip uninstall -y google-genai google-adk python-fasthtml hf-gradio gradio || true

# Install compatible versions (don't reinstall torch - use Colab's default)
#!pip install -q transformers[torch]==4.57.5 accelerate==1.4.0 peft==0.13.2 bitsandbytes==0.45.0 trl==0.9.6 datasets==3.2.0 huggingface_hub==0.25.2
!pip install -q transformers[torch]==4.57.5 accelerate==1.13.0 peft==0.19.1 bitsandbytes==0.49.2 trl==1.7.0 datasets==5.0.0 huggingface_hub==0.36.2
# gradio==5.50.0
#!pip freeze

# Verify bitsandbytes CUDA support
import bitsandbytes as bnb
print(f"bitsandbytes version: {bnb.__version__}")
#print(f"CUDA available: {bnb.cuda.is_available()}")
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("bitsandbytes backend:", getattr(bnb, "cextension", None) and bnb.cextension.BNB_BACKEND)

Wed Jul  1 18:15:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# @title 3. Login to Hugging Face & Setup Repos
from huggingface_hub import login, HfApi, create_repo
from datasets import load_dataset
import os

# Login
login(token=HF_TOKEN)
print("✅ Logged in to Hugging Face")

api = HfApi()

# Verify dataset repo exists (should be uploaded from WSL)
try:
    api.repo_info(repo_id=DATASET_REPO, repo_type="dataset")
    print(f"✅ Dataset repo exists: {DATASET_REPO}")
except Exception as e:
    print(f"❌ Dataset repo not found: {e}")
    print("Please run the WSL preparation script first:")
    print("  python scripts/prepare_and_upload_data.py")
    raise

# Create model repo if doesn't exist
try:
    api.repo_info(repo_id=MODEL_REPO, repo_type="model")
    print(f"✅ Model repo exists: {MODEL_REPO}")
except:
    create_repo(repo_id=MODEL_REPO, repo_type="model", private=True, token=HF_TOKEN)
    print(f"✅ Created private model repo: {MODEL_REPO}")

✅ Logged in to Hugging Face
✅ Dataset repo exists: COleo/cline-tools-data
✅ Model repo exists: COleo/deepseek-coder-v2-16b-tools


In [4]:
# @title 4. Load Training Data from HF Hub (Lightweight)
print("Loading datasets from HF Hub...")

train_dataset = load_dataset(DATASET_REPO, data_files="train.jsonl", split="train")
val_dataset = load_dataset(DATASET_REPO, data_files="val.jsonl", split="train")

print(f"✅ Train samples: {len(train_dataset)}")
print(f"✅ Val samples: {len(val_dataset)}")

# Inspect sample
sample = train_dataset[0]
print(f"\nSample keys: {sample.keys()}")
print(f"Sample messages: {len(sample['messages'])} messages")
for msg in sample['messages'][:2]:
    print(f"  {msg['role']}: {msg['content'][:100]}...")

Loading datasets from HF Hub...


train.jsonl:   0%|          | 0.00/16.5M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

val.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

✅ Train samples: 1080
✅ Val samples: 120

Sample keys: dict_keys(['messages'])
Sample messages: 3 messages
  system: You are a highly skilled software engineer with extensive knowledge in many programming languages, f...
  user: Integrate the new payment API by adding a client and updating the checkout flow...


In [5]:
# @title 5. Load Model with 4-bit Quantization (CUDA Context Safe)
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

print("Loading model with 4-bit quantization to GPU...")
print(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES', 'not set')} — single GPU confirmed")

# CRITICAL: Final CUDA cleanup right before model loading
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer (lightweight)
tokenizer = AutoTokenizer.from_pretrained(
    "deepseek-ai/deepseek-coder-v2-lite-base",
    trust_remote_code=True,
    padding_side="right",
    use_fast=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"✅ Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

# device_map="auto" is safe here: CUDA_VISIBLE_DEVICES=0 ensures only 1 GPU is visible,
# so accelerate will place the entire model on cuda:0 without DataParallel.
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/deepseek-coder-v2-lite-base",
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    revision="main",
)

model.config.use_cache = False
model.config.pretraining_tp = 1
print(f"✅ Model loaded: {model.__class__.__name__}")
print(f"✅ Model dtype: {next(model.parameters()).dtype}")
print(f"✅ Model device: {next(model.parameters()).device}")
print(f"GPU count visible to torch: {torch.cuda.device_count()} (must be 1)")
print(f"GPU Memory After Model Load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading model with 4-bit quantization to GPU...
CUDA_VISIBLE_DEVICES=0 — single GPU confirmed


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded. Vocab size: 100000


config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-000004.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

model-00001-of-000004.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Model loaded: DeepseekV2ForCausalLM
✅ Model dtype: torch.bfloat16
✅ Model device: cuda:0
GPU count visible to torch: 1 (must be 1)
GPU Memory After Model Load: 9.97 GB


In [6]:
# @title 6. Apply LoRA Adapters
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

# LoRA Config - Attention ONLY (not MoE experts)
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Attention only
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

print("Preparing for k-bit training...")
model = prepare_model_for_kbit_training(model)

print("Applying LoRA adapters...")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Preparing for k-bit training...
Applying LoRA adapters...
trainable params: 7,962,624 || all params: 15,714,446,848 || trainable%: 0.0507


In [7]:
# @title 7. Format Data with Chat Template
def format_chat_template(example, tokenizer):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

print("Formatting with chat template...")
train_dataset = train_dataset.map(
    lambda x: format_chat_template(x, tokenizer),
    remove_columns=train_dataset.column_names,
    num_proc=4,
)
val_dataset = val_dataset.map(
    lambda x: format_chat_template(x, tokenizer),
    remove_columns=val_dataset.column_names,
    num_proc=4,
)

print("Tokenizing...")
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=4,
)
val_dataset = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    num_proc=4,
)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

Formatting with chat template...


Map (num_proc=4):   0%|          | 0/1080 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing...


Map (num_proc=4):   0%|          | 0/1080 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/120 [00:00<?, ? examples/s]

Train samples: 1080
Val samples: 120


In [8]:
# @title 8. HF Hub Checkpoint Callback (full trainer checkpoint for resume capability)
from transformers import TrainerCallback
from huggingface_hub import HfApi
import os

class HFHubCheckpointCallback(TrainerCallback):
    """Push full trainer checkpoints to HF Hub for resume capability after runtime stops.

    Each checkpoint folder (checkpoint-N/) contains:
      - adapter_model.safetensors / adapter_config.json  (LoRA weights)
      - optimizer.pt, scheduler.pt                       (optimizer/scheduler state)
      - trainer_state.json, rng_state.pth                (training state)
    This allows trainer.train(resume_from_checkpoint=...) to fully restore training.
    """

    def __init__(self, repo_id: str, token: str, every_n_steps: int = 100):
        self.repo_id = repo_id
        self.token = token
        self.every_n_steps = every_n_steps
        self.api = HfApi(token=token)
        self.last_pushed_step = 0

    def on_save(self, args, state, control, model=None, tokenizer=None, **kwargs):
        """Called by Trainer after each local checkpoint save."""
        if state.global_step % self.every_n_steps != 0:
            return
        if state.global_step == self.last_pushed_step:
            return

        checkpoint_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        if not os.path.isdir(checkpoint_dir):
            print(f"⚠️ Checkpoint dir not found: {checkpoint_dir}. Skipping hub push.")
            return

        print(f"\n📤 Pushing full checkpoint at step {state.global_step} to HF Hub...")
        try:
            self.api.upload_folder(
                folder_path=checkpoint_dir,
                repo_id=self.repo_id,
                repo_type="model",
                path_in_repo=f"checkpoint-{state.global_step}",
                commit_message=f"Full checkpoint at step {state.global_step}",
                token=self.token,
            )
            self.last_pushed_step = state.global_step
            print(f"✅ Checkpoint pushed → https://hf.co/{self.repo_id}/tree/main/checkpoint-{state.global_step}")
        except Exception as e:
            print(f"⚠️ Failed to push checkpoint: {e}")

    def on_train_end(self, args, state, control, model=None, tokenizer=None, **kwargs):
        """Push final LoRA adapter weights after training completes."""
        print(f"\n📤 Pushing final model to HF Hub...")
        try:
            model.push_to_hub(
                self.repo_id,
                private=True,
                token=self.token,
                commit_message="Final model after training complete",
            )
            tokenizer.push_to_hub(
                self.repo_id,
                private=True,
                token=self.token,
                commit_message="Final tokenizer after training complete",
            )
            print(f"✅ Final model pushed to https://hf.co/{self.repo_id}")
        except Exception as e:
            print(f"⚠️ Failed to push final model: {e}")

# Create callback instance
hf_callback = HFHubCheckpointCallback(
    repo_id=MODEL_REPO,
    token=HF_TOKEN,
    every_n_steps=PUSH_EVERY_N_STEPS,
)
print("✅ HF Hub checkpoint callback created (uploads full trainer checkpoints)")


✅ HF Hub checkpoint callback created (uploads full trainer checkpoints)


In [9]:
# @title 9. Training Arguments (single T4 optimized)
from transformers import TrainingArguments
import os

# Environment-aware output dir (Kaggle vs Colab)
if os.path.exists("/kaggle/working"):
    OUTPUT_DIR = "/kaggle/working/checkpoints/deepseek-coder-v2-16b-tools"
else:
    OUTPUT_DIR = "/content/checkpoints/deepseek-coder-v2-16b-tools"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

_save_steps = PUSH_EVERY_N_STEPS

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=100,
    learning_rate=LEARNING_RATE,
    fp16=False,
    bf16=True,
    logging_steps=10,
    eval_strategy="no",
    save_steps=_save_steps,
    save_total_limit=3,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    seed=42,
)
print("✅ TrainingArguments set")


Output dir: /kaggle/working/checkpoints/deepseek-coder-v2-16b-tools
✅ TrainingArguments set


In [10]:
# @title 9b. Discover & Download Resume Checkpoint from HF Hub
from huggingface_hub import HfApi, snapshot_download
import os

RESUME_CHECKPOINT_PATH = None   # Will be set below if a checkpoint is found

if RESUME_FROM_HUB:
    print("Checking HF Hub for resumable checkpoints...")
    _api = HfApi(token=HF_TOKEN)
    try:
        files = list(_api.list_repo_files(repo_id=MODEL_REPO, repo_type="model", token=HF_TOKEN))

        # Full trainer checkpoints contain trainer_state.json inside checkpoint-N/ folders
        state_files = [f for f in files if f.endswith("trainer_state.json") and "/checkpoint-" in f]

        if state_files:
            # Extract step numbers from paths like "checkpoint-200/trainer_state.json"
            steps = []
            for f in state_files:
                try:
                    folder = [p for p in f.split("/") if p.startswith("checkpoint-")][0]
                    steps.append(int(folder.split("-")[1]))
                except (IndexError, ValueError):
                    pass

            if steps:
                latest_step = max(steps)
                latest_folder = f"checkpoint-{latest_step}"
                local_ckpt_dir = os.path.join(OUTPUT_DIR, latest_folder)
                print(f"Found checkpoint at step {latest_step}. Downloading to {local_ckpt_dir}...")

                snapshot_download(
                    repo_id=MODEL_REPO,
                    repo_type="model",
                    allow_patterns=f"{latest_folder}/*",
                    local_dir=OUTPUT_DIR,
                    token=HF_TOKEN,
                )
                RESUME_CHECKPOINT_PATH = local_ckpt_dir
                print(f"✅ Resume checkpoint ready: {RESUME_CHECKPOINT_PATH}")
            else:
                print("No valid checkpoint steps found. Starting fresh.")
        else:
            print("No full trainer checkpoints in HF Hub repo. Starting fresh.")
    except Exception as e:
        print(f"⚠️ Could not check for checkpoints: {e}")
        print("Starting fresh training.")
else:
    print("RESUME_FROM_HUB=False. Starting fresh training.")

print(f"Resume checkpoint: {RESUME_CHECKPOINT_PATH}")


Checking HF Hub for resumable checkpoints...
No full trainer checkpoints in HF Hub repo. Starting fresh.
Resume checkpoint: None


In [11]:
# @title 10. Initialize Trainer (WITH callback for HF Hub pushes)
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# Create trainer WITH callback for HF Hub checkpointing
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[hf_callback],  # Enabled for HF Hub checkpointing
)

print("✅ Trainer initialized with HF Hub callback")

✅ Trainer initialized with HF Hub callback


In [12]:
# @title 11. Start Training (with CUDA safety checks)
print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"GPU count: {torch.cuda.device_count()} (must be 1 — if 2, RESTART and re-run from cell 1)")

if torch.cuda.device_count() > 1:
    raise RuntimeError(
        "More than 1 GPU visible. CUDA_VISIBLE_DEVICES was not set before CUDA initialized.\n"
        "Restart the runtime and re-run ALL cells from cell 1."
    )

if torch.cuda.is_available():
    print(f"GPU Memory Before Training: {torch.cuda.memory_allocated() / 1e9:.2f} GB / "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.cuda.empty_cache()
    gc.collect()

if RESUME_CHECKPOINT_PATH:
    print(f"Resuming from checkpoint: {RESUME_CHECKPOINT_PATH}")
else:
    print("Starting fresh training (no checkpoint found).")

trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT_PATH)

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)


STARTING TRAINING
GPU count: 1 (must be 1 — if 2, RESTART and re-run from cell 1)
GPU Memory Before Training: 10.85 GB / 15.64 GB
Starting fresh training (no checkpoint found).


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.25 GiB. GPU 0 has a total capacity of 14.56 GiB of which 936.81 MiB is free. Including non-PyTorch memory, this process has 13.64 GiB memory in use. 13.83 GiB allowed; Of the allocated memory 11.59 GiB is allocated by PyTorch, and 1.93 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# @title 12. Save Final Model Locally & Push to HF Hub
print("Saving final model locally...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to: {OUTPUT_DIR}")

# Final push to HF Hub (callback also does this on_train_end)
print("Pushing final model to HF Hub...")
model.push_to_hub(
    MODEL_REPO,
    private=True,
    token=HF_TOKEN,
    commit_message="Final QLoRA adapters for Cline tools"
)
tokenizer.push_to_hub(
    MODEL_REPO,
    private=True,
    token=HF_TOKEN,
    commit_message="Final tokenizer for Cline tools"
)

print(f"✅ Final model pushed to: https://hf.co/{MODEL_REPO}")

In [ ]:
# @title 13. Zip Checkpoints for Download
import shutil
import os

# Zip all checkpoints for download
checkpoint_zip = "/content/deepseek-coder-v2-16b-qlora-checkpoints.zip"
shutil.make_archive(
    checkpoint_zip.replace('.zip', ''),
    'zip',
    OUTPUT_DIR
)
print(f"✅ Checkpoints zipped to: {checkpoint_zip}")

size_mb = os.path.getsize(checkpoint_zip) / 1e6
print(f"  Size: {size_mb:.1f} MB")

print("\n📥 Download from Colab file browser (left sidebar → Files)")
print("   Right-click → Download")

# Next Steps (Run Locally After Download)

## 1. Download Checkpoints
- From Colab file browser: `deepseek-coder-v2-16b-qlora-checkpoints.zip`
- Extract to: `ollama_tools_support/checkpoints/deepseek-coder-v2-16b-qlora/`

## 2. Merge LoRA + Convert to GGUF + Create Ollama Model
```bash
cd ollama_tools_support
source venv_ollama_tools_support/bin/activate
python scripts/merge_and_export.py
```

## 3. Test in Cline
```bash
python scripts/test_tools.py
```

## 4. Configure Cline
- Settings → Model → `deepseek-coder-v2-16b-tools`
- Enable all 9 tools
- Start coding!

---

## Resume Training After Colab Timeout
If Colab runtime stops (4h 20min limit):
1. Re-run this notebook from the top
2. It will auto-load data from HF Hub (no CPU overload)
3. It will auto-load model to GPU (no CPU overload)
4. Training continues from where it left off!

## WSL Preparation (Run Once Before Colab)
```bash
# In WSL:
cd ollama_tools_support
python scripts/prepare_and_upload_data.py
```

## Alternative: Pull Model Directly from HF Hub
```bash
# If you want to merge on a different machine:
# git clone https://hf.co/COleo/deepseek-coder-v2-16b-qlora
# Then run merge_and_export.py with that path
```